In [8]:
### RAG Pipeline- Data Ingestion to Vector DB Pipeline


In [9]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters pymupdf pypdf sentence-transformers

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached pymupdf-1.27.2.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (24 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 168.4 kB/s  0:00:06 eta 0:00:02
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 291.5 kB/s  0:00:06 eta 0:00:01
Using cached pymupdf-1.27.2.3-cp310-abi3-macosx_11_0_arm64.whl (23.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-community]ngchain-community]


In [10]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
!pip install langchain-text-splitters


/var/folders/s5/__ts1cy110d80_r4n5gsb2j40000gn/T/ipykernel_39121/1291948377.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
!pip install pypdf

In [12]:
def process_all_pdfs(pdf_directory):
    """Process all pdf files in the directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("data/pdf")

Found 3 PDF files to process

Processing: COA UNIT 6.pdf
Loaded 9 pages

Processing: unit-4 COA.pdf
Loaded 17 pages

Processing: Unit 5.pdf
Loaded 47 pages
Total documents loaded: 73


In [13]:
os.chdir("/Users/akshitasharma/RAG")
print(os.listdir("data"))

['text_file', 'pdf']


In [14]:
print(os.listdir("data/pdf"))

['COA UNIT 6.pdf', '.DS_Store', 'unit-4 COA.pdf', 'Unit 5.pdf']


In [15]:
all_pdf_documents = process_all_pdfs("data/pdf")

Found 3 PDF files to process

Processing: COA UNIT 6.pdf
Loaded 9 pages

Processing: unit-4 COA.pdf
Loaded 17 pages

Processing: Unit 5.pdf
Loaded 47 pages
Total documents loaded: 73


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf/COA UNIT 6.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'COA UNIT 6.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf/COA UNIT 6.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'COA UNIT 6.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf/COA UNIT 6.pdf', 'total_pages': 9, 'page': 2, 'page_label': '3', 'source_file': 'COA UNIT 6.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/pdf/COA UNIT 6.pdf', 'total_pages': 9, 'page': 3, 'page_label': '4', 'source_file': 'COA UNIT 6.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creato

In [17]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [18]:
chunks=split_documents(all_pdf_documents)

Split 73 documents into 49 chunks

Example chunk:
Content: Unit 5
INPUT OUTPUT...
Metadata: {'producer': 'Microsoft® PowerPoint® 2019', 'creator': 'Microsoft® PowerPoint® 2019', 'creationdate': '2025-04-01T22:42:24+05:30', 'title': '', 'author': 'meenal ratnaparkhi', 'moddate': '2025-04-01T22:42:24+05:30', 'source': 'data/pdf/Unit 5.pdf', 'total_pages': 47, 'page': 0, 'page_label': '1', 'source_file': 'Unit 5.pdf', 'file_type': 'pdf'}


import numpy as np
pip install scikit-learn
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
            
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        

In [28]:
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 5357.37it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/s5/__ts1cy110d80_r4n5gsb2j40000gn/T/ipykernel_39121/1719706246.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [29]:
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 6467.85it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/s5/__ts1cy110d80_r4n5gsb2j40000gn/T/ipykernel_39121/1719706246.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [30]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any

In [31]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [32]:
vectorstore.add_documents(chunks, embeddings)


NameError: name 'embeddings' is not defined